In [1]:
#Amo demo 1 model for each part
#GV 13.2.2026
import sys
sys.path.extend(['src', '../src']) 
# Import all ML orchestration functions
from amo.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amo.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amo.mappings import fill_quaterna_columns, learn_quaterna_mapping, reorder_midi_for_musescore
# Import amo funtion
from amo.ml_orchestration import amo

In [2]:
from amo.ml_orchestration import split_tvt_ts, split_ts, Xy_to_midi, clf_predict_back

In [3]:
import os

In [4]:
from xgboost import XGBClassifier

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support

In [6]:
#get the parent directory
cwd = os.getcwd()
parent = os.path.normpath(os.path.join(cwd, '..', '..'))
path_AMO = parent + '/AutomaticMusicOrchestration'
model_path = path_AMO + '/'
filein = path_AMO + '/data/samples/midis/symphony_5_1_orch.mid'
#filein = path_AMO + '/data/samples/midis/sugar-plum-fairy_orch.mid'
fileout = path_AMO + '/data/samples/midis/fur-elise.mid'

In [7]:
#amo will create a new file with the orchestration
#amo(file_orc,file_piano,model="XGBoost")

In [8]:
# Load and process source file
dfnmat = midi_to_dataframe(filein)
dfnmat = dfnmat.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True])
nmat = dfnmat.to_numpy()  
mapping = learn_quaterna_mapping(nmat, ytarget="track-channel")

In [9]:
# Load and process source file
dfnmat2 = midi_to_dataframe(fileout)
dfnmat2 = dfnmat2.sort_values(
        ['onset in quarter notes','duration in quarter notes', 'track number'],
        ascending=[True, True, True])
nmat2 = dfnmat2.to_numpy() 

In [10]:
X,y=defineXy(nmat) #X array (n samples,4), y array(['17_11', '16_0', '18_12', ..., '13_10', '13_10',

In [11]:
le = LabelEncoder()
le.fit(y)  # labels to integers 
y = le.transform(y)

In [12]:
X_train, X_val, X_test, y_train, y_val, y_test = split_tvt_ts(X, y, test_size=0.1) #X array (n samples,4), y array([7, 6, 8, ..., 3, 3, 3])

Partition 80/10/10.
Labels training: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Labels validation: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Labels test: [ 0  1  2  3  4  5  6  7  8  9 10 11]


In [13]:
X_train,  X_test, y_train,  y_test = split_ts(X, y, test_size=0.2)

Labels training: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Labels test: [ 0  1  2  3  4  5  6  7  8  9 10 11]


In [14]:
clf = XGBClassifier()#(tree_method="hist")
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
score = clf.score(X_test, y_test)
print("Accuracy test: %.2f%%" % (score * 100.0))
print('Accuracy train %.2f%%',% (clf.score(X_train, y_train)*100))

Accuracy val: 37.67%
Accuracy train 0.7258194519075766


In [25]:
print(precision_recall_fscore_support(y_test, y_pred, average='macro'))
print(precision_recall_fscore_support(y_test, y_pred, average='weighted'))
print(precision_recall_fscore_support(y_test, y_pred, average='micro'))

(0.4452763878715391, 0.4177593018950129, 0.3857483681892089, None)
(0.4305970961910436, 0.4021505376344086, 0.36999961515649016, None)
(0.4021505376344086, 0.4021505376344086, 0.4021505376344086, None)


In [15]:
#listen to the test set
y_pred_test = clf.predict(X_test)

In [16]:
filetest = path_AMO + '/data/samples/midis/test7.mid'
Xy_to_midi(X_test, y_pred_test, mapping, le, filetest, filein, ytarget="track-channel")
reordered = path_AMO + '/data/samples/midis/test7reorder.mid'
kept = reorder_midi_for_musescore(filetest, reordered, mapping)
print("Kept channels in output order:", kept)

Predictions map ['10_13' '11_14' '12_15' '1_0' '2_1' '3_2' '4_3' '5_4' '6_5' '7_6' '8_11'
 '9_12']

=== SAVING WITH EXACT TIMING STRUCTURE ===
ticks_per_beat: 120
Found 452 timing events:
  1. time_signature: 2/4 at 0.000 quarters (0 ticks)
  2. key_signature: Key: Eb at 0.000 quarters (0 ticks)
  3. tempo: 216.0 BPM at 0.000 quarters (0 ticks)
  4. tempo: 58.2 BPM at 2.625 quarters (315 ticks)
  5. tempo: 55.4 BPM at 2.683 quarters (322 ticks)
  6. tempo: 53.5 BPM at 2.733 quarters (328 ticks)
  7. tempo: 51.7 BPM at 2.783 quarters (334 ticks)
  8. tempo: 49.8 BPM at 2.842 quarters (341 ticks)
  9. tempo: 47.9 BPM at 2.892 quarters (347 ticks)
  10. tempo: 46.0 BPM at 2.942 quarters (353 ticks)
  11. tempo: 44.1 BPM at 3.000 quarters (360 ticks)
  12. tempo: 42.3 BPM at 3.050 quarters (366 ticks)
  13. tempo: 39.4 BPM at 3.100 quarters (372 ticks)
  14. tempo: 37.6 BPM at 3.150 quarters (378 ticks)
  15. tempo: 35.7 BPM at 3.208 quarters (385 ticks)
  16. tempo: 33.8 BPM at 3.258 quar

Kept channels in output order: [0, 1, 2, 3, 4, 5, 6, 11, 12, 13, 14, 15]
